# **Causal structure search: PC, GES, LiNGAM and NO TEARS**

Practice for the module [“Causality and Linear Models”](https://ai-interpretability.school), lesson “Causal structure search”.

In the lesson the four structure search methods were formulas and tasks on three nodes. Here we run them on data generated from a known graph with six nodes and count how many edges each of them got right. We check four claims of the lesson:

1. PC and GES recover the graph from observations only up to the Markov equivalence class: some edges stay undirected;
2. LiNGAM orients all edges if the noises are non-Gaussian and errs if they are Gaussian;
3. NO TEARS finds the graph on the raw data and loses it after the features are standardized;
4. with a small sample PC and GES err, and with a hidden confounder PC adds edges that do not exist.

Plan of the notebook:

1. the graph and the data;
2. the distance between graphs (SHD);
3. PC;
4. GES;
5. LiNGAM;
6. NO TEARS;
7. where it breaks.

The notebook runs on a CPU, the whole run takes about a minute.

In [ ]:
%pip install -q causal-learn==0.1.4.8

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import scipy.linalg as slin
import scipy.optimize as sopt
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.GES import ges
from causallearn.search.FCMBased.lingam import DirectLiNGAM
from causallearn.graph.Dag import Dag
from causallearn.graph.GraphNode import GraphNode
from causallearn.utils.DAG2CPDAG import dag2cpdag

SEED = 0

## 1. The graph and the data

Six variables are linked by a linear structural equation model $X_j = \sum_i B_{ij} X_i + e_j$:

$$X_1 \to X_2 \to X_4 \leftarrow X_3,\quad X_4 \to X_5 \to X_6 \leftarrow X_3.$$

The edge weights are chosen so that the relations along different paths do not cancel each other: otherwise the faithfulness assumption is violated, and an independence appears in the data where the graph does not promise it. We generate the data twice: with Gaussian noise $e_j \sim \mathcal N(0,1)$ and with uniform noise on $[-\sqrt3, \sqrt3]$ of the same variance. There are 2000 objects.

In [ ]:
names = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6']
d = len(names)
edges = [(0, 1, 1.0), (1, 3, 1.2), (2, 3, -1.0), (3, 4, 0.8), (4, 5, 1.1), (2, 5, -0.9)]
B = np.zeros((d, d))
for i, j, w in edges:
    B[i, j] = w                      # B[i, j] != 0: edge X_{i+1} -> X_{j+1}


def sample(n, noise, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, d))
    for j in range(d):               # the nodes are already in the order of the graph
        e = rng.normal(0, 1, n) if noise == 'gauss' else rng.uniform(-np.sqrt(3), np.sqrt(3), n)
        X[:, j] = X @ B[:, j] + e
    return X


X_gauss, X_unif = sample(2000, 'gauss'), sample(2000, 'unif')
print('variances, Gaussian noise:', X_gauss.var(0).round(2))
print('variances, uniform noise: ', X_unif.var(0).round(2))

## 2. The distance between graphs

We store a graph as a matrix $A$: $A_{ij} = 1$ is an edge $i \to j$, $A_{ij} = A_{ji} = 1$ is an undirected edge. **SHD** (structural Hamming distance) is the number of pairs of nodes whose edge differs: extra, missing, reversed, or undirected where it is directed. Zero means the graphs coincide.

PC and GES return not one graph but a Markov equivalence class, a CPDAG, where the edges whose direction the data do not determine stay undirected. So we compare their output with the CPDAG of the true graph, and the output of LiNGAM and NO TEARS with the graph itself.

In [ ]:
A_true = (B != 0).astype(int)


def to_matrix(G):
    """causal-learn graph -> matrix A: A[i, j] = 1 is an edge i -> j, A[i, j] = A[j, i] = 1 is undirected."""
    M, A = G.graph, np.zeros((d, d), int)
    for i in range(d):
        for j in range(d):
            if M[j, i] == 1 and M[i, j] == -1:
                A[i, j] = 1
            if M[i, j] == -1 and M[j, i] == -1:
                A[i, j] = A[j, i] = 1
    return A


def shd(A, A_hat):
    return sum((A[i, j], A[j, i]) != (A_hat[i, j], A_hat[j, i]) for i in range(d) for j in range(i + 1, d))


def show(A):
    out = []
    for i in range(d):
        for j in range(d):
            if A[i, j] and A[j, i] and i < j:
                out.append(f'{names[i]} - {names[j]}')
            elif A[i, j] and not A[j, i]:
                out.append(f'{names[i]} -> {names[j]}')
    return ', '.join(out)


nodes = [GraphNode(nm) for nm in names]
dag = Dag(nodes)
for i, j, _ in edges:
    dag.add_directed_edge(nodes[i], nodes[j])
A_cpdag = to_matrix(dag2cpdag(dag))
print('true graph:', show(A_true))
print('its CPDAG: ', show(A_cpdag))
print('SHD(graph, CPDAG) =', shd(A_true, A_cpdag))

The direction of the edge $X_1 \to X_2$ cannot be determined from observations: the graph with $X_2 \to X_1$ gives the same conditional independences. The other edges are oriented by two colliders, $X_2 \to X_4 \leftarrow X_3$ and $X_5 \to X_6 \leftarrow X_3$, and by Meek's rules.

## 3. PC

PC ([Spirtes and Glymour, 1991](https://doi.org/10.1177/089443939100900106)) starts from the complete undirected graph and removes the edge between $X_i$ and $X_j$ as soon as it finds a set $S$ of neighbors for which the test does not reject the conditional independence $X_i \perp X_j \mid S$. For Gaussian data the test is Fisher's z-transform of the partial correlation, `fisherz`; the significance level is $\alpha = 0.05$. Then it orients the colliders using the found sets $S$ and propagates the directions by Meek's rules.

**Task 1. Run PC on the data with Gaussian noise (`pc(X_gauss, 0.05, 'fisherz')`), convert the result to a matrix with `to_matrix` and compute the SHD to the true graph `A_true` and to its CPDAG `A_cpdag`. Give the SHD to the true graph as the answer.**

In [ ]:
# Your code here

PC found exactly the equivalence class: all edges are in place, and the only error relative to the true graph is the undirected edge $X_1 - X_2$, which is not a failure of the method but the limit of what can be learned from independences.

## 4. GES

GES ([Chickering, 2002](https://www.jmlr.org/papers/v3/chickering02b.html)) searches not by tests but by a score: it greedily adds edges while BIC grows, then greedily removes them, and it does so in the space of equivalence classes. Its output is also a CPDAG.

In [ ]:
A_ges = to_matrix(ges(X_gauss, score_func='local_score_BIC')['G'])
print('GES:', show(A_ges))
print(f'SHD to the true graph {shd(A_true, A_ges)} | to the CPDAG {shd(A_cpdag, A_ges)}')

## 5. LiNGAM

LiNGAM ([Shimizu et al., 2006](https://www.jmlr.org/papers/v7/shimizu06a.html)) uses the fact that with non-Gaussian noise the regression in the right direction leaves a residual independent of the regressor, and in the wrong direction it does not. DirectLiNGAM ([Shimizu et al., 2011](https://www.jmlr.org/papers/v12/shimizu11a.html)) finds, one by one, a variable independent of the residuals of all regressions on it, puts it next in the order and subtracts it. The output is a graph with the directions of all edges and weights. We treat weights below 0.1 in absolute value as zero.

**Task 2. Run `DirectLiNGAM` on the data with uniform and with Gaussian noise. The weight matrix is `model.adjacency_matrix_`; in it a row is the effect and a column is the cause, so it has to be transposed. Compute the SHD to the true graph in both cases. Give the SHD for the Gaussian noise as the answer.**

In [ ]:
# Your code here

With uniform noise LiNGAM found the whole graph, including the direction $X_1 \to X_2$ that PC and GES could not determine. With Gaussian noise the regression residuals in both directions are equally independent of the regressor, and the order DirectLiNGAM chooses no longer reflects the graph.

## 6. NO TEARS

NO TEARS ([Zheng et al., 2018](https://arxiv.org/abs/1803.01422)) solves the problem

$$\min_W \frac{1}{2n}\|X - XW\|_F^2 + \lambda\|W\|_1 \quad \text{s.t.} \quad h(W) = \operatorname{tr}\big(e^{W\circ W}\big) - d = 0$$

by the augmented Lagrangian method. Below is the linear version from the [authors' repository](https://github.com/xunzheng/notears) (Apache-2.0 license) almost unchanged; the penalty is $\lambda = 0.1$, weights below 0.3 in absolute value are set to zero.

In [ ]:
def notears_linear(X, lambda1=0.1, max_iter=100, h_tol=1e-8, rho_max=1e16, w_threshold=0.3):
    """Linear NO TEARS after notears/linear.py (Zheng et al., 2018; Apache-2.0), squared loss."""
    n, d = X.shape
    X = X - X.mean(axis=0, keepdims=True)

    def _h(W):
        E = slin.expm(W * W)
        return np.trace(E) - d, E.T * W * 2

    def _adj(w):
        return (w[:d * d] - w[d * d:]).reshape(d, d)

    def _func(w):
        W = _adj(w)
        R = X - X @ W
        loss, G_loss = 0.5 / n * (R ** 2).sum(), -1.0 / n * X.T @ R
        h, G_h = _h(W)
        obj = loss + 0.5 * rho * h * h + alpha * h + lambda1 * w.sum()
        G = G_loss + (rho * h + alpha) * G_h
        return obj, np.concatenate((G + lambda1, -G + lambda1), axis=None)

    w_est, rho, alpha, h = np.zeros(2 * d * d), 1.0, 0.0, np.inf
    bnds = [(0, 0) if i == j else (0, None) for _ in range(2) for i in range(d) for j in range(d)]
    for _ in range(max_iter):
        while rho < rho_max:
            w_new = sopt.minimize(_func, w_est, method='L-BFGS-B', jac=True, bounds=bnds).x
            h_new = _h(_adj(w_new))[0]
            if h_new > 0.25 * h:
                rho *= 10
            else:
                break
        w_est, h = w_new, h_new
        alpha += rho * h
        if h <= h_tol or rho >= rho_max:
            break
    W = _adj(w_est)
    W[np.abs(W) < w_threshold] = 0
    return W


A_nt = (notears_linear(X_gauss) != 0).astype(int)
print(f'NO TEARS on the raw data: SHD {shd(A_true, A_nt)} | {show(A_nt)}')

On the raw data with Gaussian noise NO TEARS found the whole graph with directions, although PC and GES could not determine the direction $X_1 \to X_2$. The reason is not the method but the data: [Reisach, Seiler, and Weichwald, 2021](https://arxiv.org/abs/2102.13647) noticed that in such simulations the variance grows along the graph, the effect having a larger variance than the cause, and the squared loss picks the direction “from smaller variance to larger”. They called this property varsortability.

**Task 3. On `X_gauss`, compute the share of edges of the true graph where the variance of the effect is larger than the variance of the cause: a simplified version of varsortability. Round the answer to three decimal places.**

In [ ]:
# Your code here

The variance grows along five edges of six, and NO TEARS uses it.

**Task 4. Standardize the features of `X_gauss` (subtract the mean, divide by the standard deviation), run `notears_linear` and compute the SHD to the true graph.**

In [ ]:
# Your code here

After standardization the hint disappears, and NO TEARS errs in five pairs of nodes: four edges reversed, one missing. PC and GES do not depend on scale: partial correlations and the BIC of a linear Gaussian model do not change when a feature is multiplied by a number.

In [ ]:
X_std = (X_gauss - X_gauss.mean(0)) / X_gauss.std(0)
shd_pc_std = shd(A_cpdag, to_matrix(pc(X_std, 0.05, 'fisherz', show_progress=False).G))
shd_ges_std = shd(A_cpdag, to_matrix(ges(X_std, score_func='local_score_BIC')['G']))
print(f'PC: SHD {shd_pc_std} | GES: SHD {shd_ges_std}')

## 7. Where it breaks

### 7.1. Too little data

PC and GES on Gaussian data of different sizes, five random samples per size; the table shows the mean SHD to the CPDAG.

In [ ]:
rows = []
for n in (100, 200, 500, 1000, 2000):
    shd_pc, shd_ges = [], []
    for s in range(5):
        Xn = sample(n, 'gauss', seed=s)
        shd_pc.append(shd(A_cpdag, to_matrix(pc(Xn, 0.05, 'fisherz', show_progress=False).G)))
        shd_ges.append(shd(A_cpdag, to_matrix(ges(Xn, score_func='local_score_BIC')['G'])))
    rows.append(dict(n=n, PC=np.mean(shd_pc), GES=np.mean(shd_ges)))
pd.DataFrame(rows)

With 100–1000 objects PC errs in 1.6–3.8 pairs of nodes on average: the test does not reject a weak dependence, the edge is removed, and the colliders are then oriented wrongly. GES errs less often on these data, but it does not always find the equivalence class either.

### 7.2. A hidden confounder

Remove $X_3$, the common cause of $X_4$ and $X_6$, from the data and run PC on the remaining five variables.

In [ ]:
keep = [0, 1, 3, 4, 5]
G_hidden = pc(X_gauss[:, keep], 0.05, 'fisherz', show_progress=False, node_names=[names[k] for k in keep]).G
print(G_hidden)

PC added the edges $X_4 \to X_6$ and $X_2 \to X_6$, which are not in the graph: without $X_3$ the dependence of $X_4$ and $X_6$ is explained by nothing but a direct edge. PC, GES, LiNGAM and NO TEARS in this notebook assume that all common causes are measured. For data with hidden confounders there is FCI, a variant of PC that marks the edges behind which an unmeasured cause may stand (`causallearn.search.ConstraintBased.FCI.fci`).

## What to take away

- **PC and GES find the equivalence class, not the graph.** On 2000 objects both found the CPDAG without errors but could not direct the edge $X_1 - X_2$: it cannot be directed from independences at all.
- **LiNGAM directs all edges if the noise is non-Gaussian.** With uniform noise SHD is 0, with Gaussian noise it is 8.
- **NO TEARS wins on simulations thanks to the scale.** The variance grows along five edges of six; after standardization SHD grows from 0 to 5, while PC and GES do not change on the standardized data.
- **A found graph is a hypothesis.** With a small sample the errors cascade, and a hidden confounder adds edges that do not exist.

Sources:
1. [Spirtes and Glymour, 1991. An Algorithm for Fast Recovery of Sparse Causal Graphs](https://doi.org/10.1177/089443939100900106)
2. [Chickering, 2002. Optimal Structure Identification With Greedy Search](https://www.jmlr.org/papers/v3/chickering02b.html)
3. [Shimizu, Hoyer, Hyvärinen, and Kerminen, 2006. A Linear Non-Gaussian Acyclic Model for Causal Discovery](https://www.jmlr.org/papers/v7/shimizu06a.html)
4. [Shimizu et al., 2011. DirectLiNGAM: A Direct Method for Learning a Linear Non-Gaussian Structural Equation Model](https://www.jmlr.org/papers/v12/shimizu11a.html)
5. [Zheng, Aragam, Ravikumar, and Xing, 2018. DAGs with NO TEARS: Continuous Optimization for Structure Learning](https://arxiv.org/abs/1803.01422)
6. [Reisach, Seiler, and Weichwald, 2021. Beware of the Simulated DAG! Causal Discovery Benchmarks May Be Easy To Game](https://arxiv.org/abs/2102.13647)
7. [Meek, 1995. Causal Inference and Causal Explanation with Background Knowledge](https://arxiv.org/abs/1302.4972)